# Interactive Scoring Explorer

**Companion to:** *"Skill, Sequence, and Scoring"* — Michael Borck, Curtin University

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michael-borck/skill-sequence-scoring/blob/main/notebooks/02_interactive_explorer.ipynb)

Explore the differences between traditional and World Bowling scoring interactively. Build your own frame sequences, adjust skill parameters, and see the effects live.

In [ ]:
# Setup — run this cell first
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, IntSlider, FloatSlider, Dropdown, VBox, HBox, Output, Label
from IPython.display import display, HTML

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
})
TC, WC = '#1a1a1a', '#888888'

def score_traditional(balls):
    score, i = 0, 0
    for frame in range(10):
        if i >= len(balls): return None
        if frame < 9:
            if balls[i] == 10:
                if i + 2 >= len(balls): return None
                score += 10 + balls[i+1] + balls[i+2]; i += 1
            else:
                if i + 1 >= len(balls): return None
                if balls[i] + balls[i+1] > 10: return None
                score += balls[i] + balls[i+1]
                if balls[i] + balls[i+1] == 10:
                    if i + 2 >= len(balls): return None
                    score += balls[i+2]
                i += 2
        else:
            if balls[i] == 10:
                if i + 2 >= len(balls): return None
                score += 10 + balls[i+1] + balls[i+2]
            else:
                if i + 1 >= len(balls): return None
                if balls[i] + balls[i+1] > 10: return None
                if balls[i] + balls[i+1] == 10:
                    if i + 2 >= len(balls): return None
                    score += 10 + balls[i+2]
                else:
                    score += balls[i] + balls[i+1]
    return score

def score_world(balls):
    score, i = 0, 0
    for frame in range(10):
        if i >= len(balls): return None
        if balls[i] == 10:
            score += 30; i += 1
        else:
            if i + 1 >= len(balls): return None
            if balls[i] + balls[i+1] > 10: return None
            if balls[i] + balls[i+1] == 10: score += 10 + balls[i]
            else: score += balls[i] + balls[i+1]
            i += 2
    return score

print("Setup complete ✓")

## 1. Reward Gradient Explorer

Adjust the number of consecutive strikes and the non-strike fill to see how the marginal value changes.

In [ ]:
def reward_gradient(fill_b1=5, fill_b2=4):
    """Show marginal value of each consecutive strike for a given fill."""
    def game(n):
        balls = [10]*n + [fill_b1, fill_b2]*(10-n)
        if n == 10: balls = [10]*12
        elif n == 9: balls = [10]*9 + [10, fill_b1, fill_b2]
        return balls

    ts = [score_traditional(game(n)) for n in range(11)]
    ws = [score_world(game(n)) for n in range(11)]
    tm = [ts[i]-ts[i-1] for i in range(1, 11)]
    wm = [ws[i]-ws[i-1] for i in range(1, 11)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(range(11), ts, '-o', color=TC, label='Traditional', markersize=6)
    ax1.plot(range(11), ws, '--s', color=WC, label='World Bowling', markersize=6)
    ax1.set_xlabel('Consecutive Strikes'); ax1.set_ylabel('Total Score')
    ax1.set_title(f'Total Score (fill: {fill_b1},{fill_b2})'); ax1.legend(); ax1.set_xticks(range(11))

    x = list(range(1, 11))
    ax2.bar([i-0.18 for i in x], tm, 0.35, color='#2c2c2c', edgecolor=TC, alpha=0.85, label='Traditional')
    ax2.bar([i+0.18 for i in x], wm, 0.35, color='#aaa', edgecolor=WC, hatch='///', alpha=0.85, label='World Bowling')
    ax2.axhline(wm[0], color=WC, linestyle=':', alpha=0.5)
    ax2.set_xlabel('Strike #'); ax2.set_ylabel('Marginal Increase')
    ax2.set_title('Marginal Value'); ax2.legend(); ax2.set_xticks(x)
    fig.tight_layout(); plt.show()

    print(f"10th strike marginal: Traditional +{tm[9]}, World Bowling +{wm[9]}")
    print(f"1st strike marginal:  Traditional +{tm[0]}, World Bowling +{wm[0]}")

interact(reward_gradient,
         fill_b1=IntSlider(min=0, max=9, value=5, description='Fill ball 1:'),
         fill_b2=IntSlider(min=0, max=9, value=4, description='Fill ball 2:'));

## 2. Sequence Sensitivity Explorer

Choose a composition (number of strikes and spares) and see all possible orderings scored under both systems. Notice: World Bowling score is always the same regardless of order.

In [ ]:
from itertools import permutations
from collections import Counter

def unique_perms(items):
    seen = set()
    for p in permutations(items):
        if p not in seen: seen.add(p); yield p

def sequence_explorer(n_strikes=5):
    n_spares = 9 - n_strikes
    strike, spare, n10 = 10, (5,5), [5,4]

    frames = [strike]*n_strikes + [spare]*n_spares
    trad_scores, world_scores = [], []

    for perm in unique_perms(tuple(frames)):
        balls = []
        for f in perm:
            if isinstance(f, tuple): balls.extend(f)
            else: balls.append(f)
        balls.extend(n10)
        t, w = score_traditional(balls), score_world(balls)
        if t and w: trad_scores.append(t); world_scores.append(w)

    n_perms = len(trad_scores)
    tc = Counter(trad_scores)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    tx = sorted(tc.keys()); ty = [tc[s] for s in tx]
    ax.bar(tx, ty, width=0.8, color='#2c2c2c', edgecolor=TC, alpha=0.85, label='Traditional')
    wb = world_scores[0]
    ax.axvline(wb, color=WC, linewidth=2.5, linestyle='--', label=f'World Bowling (all = {wb})')
    ax.set_xlabel('Score'); ax.set_ylabel('Number of Orderings')
    ax.set_title(f'{n_strikes} Strikes + {n_spares} Spares(5,5) — {n_perms} orderings')
    ax.legend(); plt.show()

    print(f"Traditional: range {min(trad_scores)}–{max(trad_scores)} "
          f"({max(trad_scores)-min(trad_scores)} pts spread)")
    print(f"World Bowling: always {wb} (0 pts spread)")

interact(sequence_explorer,
         n_strikes=IntSlider(min=1, max=8, value=5, description='Strikes:'));

## 3. Skill Tier Simulator

Adjust strike rate and spare rate to simulate games at any skill level. Compare score distributions and spread under both systems.

In [ ]:
def sim_first(rng, ps, pm):
    if rng.random() < ps: return 10
    p = min(pm/10, 0.95); pins = rng.binomial(10, p)
    while pins == 10: pins = rng.binomial(10, p)
    return int(pins)

def sim_second(rng, b1, psp):
    rem = 10-b1
    if rem == 0: return 0
    return rem if rng.random() < psp else int(rng.integers(0, rem))

def sim_game_fn(rng, ps, psp, pm):
    balls = []
    for _ in range(9):
        b1 = sim_first(rng, ps, pm); balls.append(b1)
        if b1 < 10: balls.append(sim_second(rng, b1, psp))
    b1 = sim_first(rng, ps, pm); balls.append(b1)
    if b1 == 10:
        b2 = sim_first(rng, ps, pm); balls.append(b2)
        balls.append(sim_first(rng, ps, pm) if b2 == 10 else sim_second(rng, b2, psp))
    else:
        b2 = sim_second(rng, b1, psp); balls.append(b2)
        if b1+b2 == 10: balls.append(sim_first(rng, ps, pm))
    return balls

def skill_sim(strike_pct=40, spare_pct=60, n_games=10000):
    ps, psp = strike_pct/100, spare_pct/100
    pm = min(4.5 + ps*5, 9.0)
    rng = np.random.default_rng(42)
    ts, ws = [], []
    for _ in range(n_games):
        balls = sim_game_fn(rng, ps, psp, pm)
        t, w = score_traditional(balls), score_world(balls)
        if t and w: ts.append(t); ws.append(w)
    ts, ws = np.array(ts), np.array(ws)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    bins = np.arange(0, 305, 5)
    ax1.hist(ts, bins=bins, alpha=0.7, color='#2c2c2c', edgecolor=TC, label='Traditional', density=True)
    ax1.hist(ws, bins=bins, alpha=0.5, color='#aaa', edgecolor=WC, hatch='///', label='World Bowling', density=True)
    ax1.set_xlabel('Score'); ax1.set_ylabel('Density')
    ax1.set_title(f'Score Distributions ({strike_pct}% strike rate)'); ax1.legend()

    data = {'Traditional': ts, 'World Bowling': ws}
    labels = list(data.keys())
    means = [np.mean(d) for d in data.values()]
    sds = [np.std(d) for d in data.values()]
    x = [0, 1]
    ax2.bar(x, sds, color=[TC, WC], alpha=0.7, width=0.5)
    ax2.set_xticks(x); ax2.set_xticklabels(labels)
    ax2.set_ylabel('Score SD'); ax2.set_title('Score Spread Comparison')
    for i, (m, s) in enumerate(zip(means, sds)):
        ax2.text(i, s+0.3, f'Mean={m:.0f}\nSD={s:.1f}', ha='center', fontsize=9)
    fig.tight_layout(); plt.show()

interact(skill_sim,
         strike_pct=IntSlider(min=2, max=85, value=40, description='Strike %:'),
         spare_pct=IntSlider(min=10, max=95, value=60, description='Spare %:'),
         n_games=Dropdown(options=[1000, 5000, 10000, 30000], value=10000, description='Games:'));

## 4. Quick Score Calculator

Enter a ball sequence and see both scores instantly. Use this to verify any example from the paper.

**Format:** comma-separated pin counts. A strike is `10`. A spare is two numbers summing to 10 (e.g., `7,3`).

**Examples:**
- Perfect game: `10,10,10,10,10,10,10,10,10,10,10,10`
- All spares (5,5): `5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5`
- 5X then 4sp then open: `10,10,10,10,10,5,5,5,5,5,5,5,5,5,4`

In [ ]:
def calc_score(ball_sequence="10,10,10,10,10,5,5,5,5,5,5,5,5,5,4"):
    """Parse a comma-separated ball sequence and score it."""
    try:
        balls = [int(x.strip()) for x in ball_sequence.split(',')]
    except ValueError:
        print("❌ Invalid input — use comma-separated integers (e.g., 10,5,5,8,1)")
        return

    t = score_traditional(balls)
    w = score_world(balls)

    print(f"Balls: {balls}")
    print(f"{'─'*40}")
    print(f"Traditional:  {t if t is not None else 'INVALID SEQUENCE'}")
    print(f"World Bowling: {w if w is not None else 'INVALID SEQUENCE'}")
    if t is not None and w is not None:
        diff = t - w
        print(f"Difference:   {'+' if diff > 0 else ''}{diff} "
              f"({'Traditional higher' if diff > 0 else 'World Bowling higher' if diff < 0 else 'Equal'})")

# Some examples from the paper
examples = [
    ("Perfect game", "10,10,10,10,10,10,10,10,10,10,10,10"),
    ("5X then 4sp + open", "10,10,10,10,10,5,5,5,5,5,5,5,5,5,4"),
    ("4sp then 5X + open", "5,5,5,5,5,5,5,5,10,10,10,10,10,5,4"),
    ("Alt X/sp ×4 then X + open", "10,5,5,10,5,5,10,5,5,10,5,5,10,5,4"),
    ("All spares (5,5) + 5", "5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5"),
]

print("Paper examples:\n")
for name, seq in examples:
    balls = [int(x) for x in seq.split(',')]
    t, w = score_traditional(balls), score_world(balls)
    print(f"  {name:35s}  Trad={t:>3}  WB={w:>3}  Diff={t-w:>+3}")

print("\n\nTry your own below — edit the string and re-run:")
calc_score("10,10,10,10,10,5,5,5,5,5,5,5,5,5,4")